# 📊 Phase 2 — Exploratory Data Analysis (EDA)
**Project**: Blinkit Supply Chain & Inventory Optimization  
**Author**: Rishi Verma

---
### 🎯 Goals:
- Sales distribution by category, outlet, location
- Inventory trends — stock received vs damaged
- Delivery performance analysis
- Customer segmentation overview
- Marketing ROI analysis
- Correlation & heatmap analysis
---

## 1️⃣ Import Libraries & Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings, os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

# Style
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
sns.set_palette('Set2')

CHARTS = '../outputs/charts'
os.makedirs(CHARTS, exist_ok=True)

# Load cleaned data
CLEANED = '../data/cleaned'
products   = pd.read_csv(f'{CLEANED}/products_cleaned.csv')
inventory  = pd.read_csv(f'{CLEANED}/inventory_cleaned.csv', parse_dates=['date'])
orders     = pd.read_csv(f'{CLEANED}/orders_cleaned.csv',    parse_dates=['order_date'])
order_items= pd.read_csv(f'{CLEANED}/order_items_cleaned.csv')
customers  = pd.read_csv(f'{CLEANED}/customers_cleaned.csv')
delivery   = pd.read_csv(f'{CLEANED}/delivery_cleaned.csv')
feedback   = pd.read_csv(f'{CLEANED}/feedback_cleaned.csv')
marketing  = pd.read_csv(f'{CLEANED}/marketing_cleaned.csv', parse_dates=['date'])
master_sales = pd.read_csv(f'{CLEANED}/master_sales.csv',    parse_dates=['order_date'])
master_inv   = pd.read_csv(f'{CLEANED}/master_inventory.csv',parse_dates=['date'])

print('✅ All cleaned datasets loaded!')
print(f'master_sales    : {master_sales.shape}')
print(f'master_inventory: {master_inv.shape}')

## 2️⃣ 📌 KPI Summary Dashboard

In [ ]:
total_revenue   = orders['order_total'].sum()
total_orders    = orders['order_id'].nunique()
avg_order_val   = orders['order_total'].mean()
total_products  = products['product_id'].nunique()
total_customers = customers['customer_id'].nunique()
late_pct        = orders['is_late'].mean() * 100
total_damaged   = inventory['damaged_stock'].sum()
avg_damage_rate = inventory['damage_rate'].mean()

print('=' * 55)
print('           📊 BLINKIT KPI DASHBOARD')
print('=' * 55)
print(f'  💰 Total Revenue        : ₹{total_revenue:>12,.0f}')
print(f'  🛒 Total Orders         : {total_orders:>12,}')
print(f'  📦 Avg Order Value      : ₹{avg_order_val:>12,.2f}')
print(f'  🏷️  Unique Products      : {total_products:>12,}')
print(f'  👥 Total Customers      : {total_customers:>12,}')
print(f'  🚴 Late Deliveries      : {late_pct:>11.1f}%')
print(f'  💥 Total Damaged Stock  : {total_damaged:>12,}')
print(f'  📉 Avg Damage Rate      : {avg_damage_rate:>11.2f}%')
print('=' * 55)

## 3️⃣ 🛒 Sales Analysis

In [ ]:
# ── Revenue by Category ──
cat_revenue = master_sales.groupby('category')['line_total'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart
bars = axes[0].barh(cat_revenue.index, cat_revenue.values / 1e3,
                    color=sns.color_palette('Set2', len(cat_revenue)))
axes[0].set_xlabel('Revenue (₹ Thousands)')
axes[0].set_title('💰 Revenue by Product Category', fontsize=14, fontweight='bold')
for bar, val in zip(bars, cat_revenue.values):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'₹{val/1e3:.0f}K', va='center', fontsize=9)

# Pie chart
axes[1].pie(cat_revenue.values, labels=cat_revenue.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(cat_revenue)))
axes[1].set_title('📊 Revenue Share by Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CHARTS}/01_revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 01_revenue_by_category.png')

In [ ]:
# ── Monthly Revenue Trend ──
monthly = orders.groupby(['order_year', 'order_month', 'order_month_name'])['order_total'].sum().reset_index()
monthly = monthly.sort_values(['order_year', 'order_month'])
monthly['period'] = monthly['order_month_name'] + ' ' + monthly['order_year'].astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(monthly['period'], monthly['order_total'] / 1e3,
        marker='o', linewidth=2.5, markersize=8, color='#2ecc71')
ax.fill_between(range(len(monthly)), monthly['order_total'] / 1e3,
                alpha=0.15, color='#2ecc71')
ax.set_title('📈 Monthly Revenue Trend', fontsize=15, fontweight='bold')
ax.set_ylabel('Revenue (₹ Thousands)')
ax.set_xlabel('Month')
plt.xticks(range(len(monthly)), monthly['period'], rotation=45, ha='right')
for i, row in monthly.iterrows():
    ax.annotate(f"₹{row['order_total']/1e3:.0f}K",
                xy=(list(monthly['period']).index(row['period']), row['order_total']/1e3),
                xytext=(0, 10), textcoords='offset points', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{CHARTS}/02_monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 02_monthly_revenue_trend.png')

In [ ]:
# ── Orders by Day of Week & Hour ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = orders['order_day'].value_counts().reindex(day_order)
axes[0].bar(day_counts.index, day_counts.values,
            color=sns.color_palette('coolwarm', 7))
axes[0].set_title('📅 Orders by Day of Week', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=30)

hour_counts = orders['order_hour'].value_counts().sort_index()
axes[1].bar(hour_counts.index, hour_counts.values,
            color=sns.color_palette('viridis', 24))
axes[1].set_title('⏰ Orders by Hour of Day', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Orders')
axes[1].set_xlabel('Hour (24h)')

plt.tight_layout()
plt.savefig(f'{CHARTS}/03_orders_by_day_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 03_orders_by_day_hour.png')

In [ ]:
# ── Top 15 Best Selling Products ──
top_products = (master_sales.groupby('product_name')['quantity']
                .sum().sort_values(ascending=False).head(15))

fig, ax = plt.subplots(figsize=(14, 7))
colors = sns.color_palette('RdYlGn_r', 15)
bars = ax.barh(top_products.index[::-1], top_products.values[::-1], color=colors)
ax.set_title('🏆 Top 15 Best-Selling Products (by Quantity)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Quantity Sold')
for bar, val in zip(bars, top_products.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/04_top_15_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 04_top_15_products.png')

## 4️⃣ 📦 Inventory Analysis

In [ ]:
# ── Daily Stock Received vs Damaged Over Time ──
daily_inv = inventory.groupby('date').agg(
    stock_received=('stock_received', 'sum'),
    damaged_stock=('damaged_stock', 'sum')
).reset_index()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily_inv['date'], daily_inv['stock_received'],
        label='📦 Stock Received', color='#3498db', linewidth=1.5)
ax.plot(daily_inv['date'], daily_inv['damaged_stock'],
        label='💥 Damaged Stock', color='#e74c3c', linewidth=1.5)
ax.fill_between(daily_inv['date'], daily_inv['damaged_stock'],
                alpha=0.2, color='#e74c3c')
ax.set_title('📦 Daily Stock Received vs Damaged Stock', fontsize=14, fontweight='bold')
ax.set_ylabel('Units')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{CHARTS}/05_stock_received_vs_damaged.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 05_stock_received_vs_damaged.png')

In [ ]:
# ── Damage Rate by Category ──
cat_damage = master_inv.groupby('category').agg(
    total_received=('stock_received', 'sum'),
    total_damaged=('damaged_stock', 'sum')
).reset_index()
cat_damage['damage_rate'] = (cat_damage['total_damaged'] / cat_damage['total_received'] * 100).round(2)
cat_damage = cat_damage.sort_values('damage_rate', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#e74c3c' if r > cat_damage['damage_rate'].mean() else '#2ecc71'
          for r in cat_damage['damage_rate']]
bars = ax.bar(cat_damage['category'], cat_damage['damage_rate'], color=colors)
ax.axhline(cat_damage['damage_rate'].mean(), color='orange', linestyle='--',
           linewidth=2, label=f"Avg: {cat_damage['damage_rate'].mean():.1f}%")
ax.set_title('⚠️ Damage Rate by Product Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Damage Rate (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
for bar, val in zip(bars, cat_damage['damage_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/06_damage_rate_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 06_damage_rate_by_category.png')

In [ ]:
# ── Perishable vs Non-Perishable Stock Analysis ──
perish = products.groupby('is_perishable').agg(
    count=('product_id', 'count'),
    avg_shelf_life=('shelf_life_days', 'mean'),
    avg_margin=('margin_percentage', 'mean'),
    avg_price=('price', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].pie(perish['count'], labels=perish['is_perishable'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[0].set_title('Perishable vs Non-Perishable\n(Product Count)', fontweight='bold')

axes[1].bar(perish['is_perishable'], perish['avg_margin'],
            color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Avg Margin % by Perishability', fontweight='bold')
axes[1].set_ylabel('Avg Margin (%)')

axes[2].bar(perish['is_perishable'], perish['avg_price'],
            color=['#e74c3c', '#2ecc71'])
axes[2].set_title('Avg Price by Perishability', fontweight='bold')
axes[2].set_ylabel('Avg Price (₹)')

plt.suptitle('🥦 Perishable vs Non-Perishable Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/07_perishable_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 07_perishable_analysis.png')

## 5️⃣ 🚴 Delivery Performance Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Delivery Status Distribution
status_counts = orders['delivery_status'].value_counts()
axes[0].pie(status_counts.values, labels=status_counts.index,
            autopct='%1.1f%%', colors=['#2ecc71','#e74c3c','#f39c12'],
            startangle=90)
axes[0].set_title('🚴 Delivery Status Split', fontweight='bold')

# Delivery Time Distribution
axes[1].hist(delivery['delivery_time_minutes'], bins=30,
             color='#3498db', edgecolor='white', alpha=0.8)
axes[1].axvline(delivery['delivery_time_minutes'].mean(), color='red',
                linestyle='--', label=f"Mean: {delivery['delivery_time_minutes'].mean():.1f} min")
axes[1].set_title('⏱️ Delivery Time Distribution', fontweight='bold')
axes[1].set_xlabel('Minutes')
axes[1].legend()

# Delay reasons
if 'reasons_if_delayed' in delivery.columns:
    delay_reasons = delivery['reasons_if_delayed'].dropna().value_counts().head(8)
    axes[2].barh(delay_reasons.index, delay_reasons.values, color='#e74c3c')
    axes[2].set_title('❌ Top Delay Reasons', fontweight='bold')
    axes[2].set_xlabel('Count')

plt.suptitle('🚴 Delivery Performance Deep Dive', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/08_delivery_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 08_delivery_performance.png')

## 6️⃣ 👥 Customer Segmentation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Segment count
seg = customers['customer_segment'].value_counts()
axes[0].bar(seg.index, seg.values,
            color=sns.color_palette('Set2', len(seg)))
axes[0].set_title('👥 Customers by Segment', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(seg.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Avg order value by segment
seg_aov = customers.groupby('customer_segment')['avg_order_value'].mean().sort_values(ascending=False)
axes[1].bar(seg_aov.index, seg_aov.values,
            color=sns.color_palette('Set1', len(seg_aov)))
axes[1].set_title('💰 Avg Order Value by Segment', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Avg Order Value (₹)')
for i, v in enumerate(seg_aov.values):
    axes[1].text(i, v + 5, f'₹{v:.0f}', ha='center', fontweight='bold')

plt.suptitle('👥 Customer Segmentation Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/09_customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 09_customer_segmentation.png')

## 7️⃣ ⭐ Customer Feedback & Sentiment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Rating Distribution
rating_counts = feedback['rating'].value_counts().sort_index()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
axes[0].bar(rating_counts.index, rating_counts.values, color=colors)
axes[0].set_title('⭐ Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Rating (1-5)')
axes[0].set_ylabel('Count')

# Sentiment Distribution
sentiment = feedback['sentiment'].value_counts()
axes[1].pie(sentiment.values, labels=sentiment.index,
            autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c','#f39c12'], startangle=90)
axes[1].set_title('😊 Sentiment Distribution', fontweight='bold')

# Feedback Category
fb_cat = feedback['feedback_category'].value_counts().head(8)
axes[2].barh(fb_cat.index, fb_cat.values, color=sns.color_palette('Set3', len(fb_cat)))
axes[2].set_title('📝 Feedback Categories', fontweight='bold')
axes[2].set_xlabel('Count')

plt.suptitle('⭐ Customer Feedback & Sentiment Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/10_feedback_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 10_feedback_sentiment.png')

## 8️⃣ 📣 Marketing Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ROAS by Channel
roas_channel = marketing.groupby('channel')['roas'].mean().sort_values(ascending=False)
axes[0].bar(roas_channel.index, roas_channel.values,
            color=sns.color_palette('Set2', len(roas_channel)))
axes[0].set_title('📣 Avg ROAS by Channel', fontweight='bold')
axes[0].set_ylabel('ROAS')
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(1, color='red', linestyle='--', label='Break-even (ROAS=1)')
axes[0].legend()

# CTR by Channel
ctr_channel = marketing.groupby('channel')['ctr'].mean().sort_values(ascending=False)
axes[1].bar(ctr_channel.index, ctr_channel.values,
            color=sns.color_palette('Set1', len(ctr_channel)))
axes[1].set_title('🖱️ Avg CTR by Channel (%)', fontweight='bold')
axes[1].set_ylabel('CTR (%)')
axes[1].tick_params(axis='x', rotation=30)

# Spend vs Revenue
mkt_sum = marketing.groupby('channel')[['spend','revenue_generated']].sum()
x = range(len(mkt_sum))
w = 0.35
axes[2].bar([i - w/2 for i in x], mkt_sum['spend'] / 1e3,
            w, label='Spend (₹K)', color='#e74c3c')
axes[2].bar([i + w/2 for i in x], mkt_sum['revenue_generated'] / 1e3,
            w, label='Revenue (₹K)', color='#2ecc71')
axes[2].set_xticks(x)
axes[2].set_xticklabels(mkt_sum.index, rotation=30)
axes[2].set_title('💸 Spend vs Revenue by Channel', fontweight='bold')
axes[2].set_ylabel('₹ Thousands')
axes[2].legend()

plt.suptitle('📣 Marketing Performance Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/11_marketing_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 11_marketing_performance.png')

## 9️⃣ 🔥 Correlation Heatmap

In [ ]:
# Numeric columns from master_sales
numeric_cols = ['quantity','unit_price','line_total','price','mrp',
                'margin_percentage','shelf_life_days','min_stock_level',
                'max_stock_level','discount_pct','stock_range']
corr_cols = [c for c in numeric_cols if c in master_sales.columns]
corr_matrix = master_sales[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('🔥 Correlation Heatmap — Sales & Inventory Features',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{CHARTS}/12_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 12_correlation_heatmap.png')

## 🔟 EDA Summary & Key Insights

In [ ]:
print('=' * 60)
print('        📊 PHASE 2 — EDA KEY INSIGHTS')
print('=' * 60)

top_cat = master_sales.groupby('category')['line_total'].sum().idxmax()
top_prod = master_sales.groupby('product_name')['quantity'].sum().idxmax()
best_channel = marketing.groupby('channel')['roas'].mean().idxmax()
worst_cat_dmg = cat_damage.iloc[0]['category']

print(f"  🏆 Top Revenue Category  : {top_cat}")
print(f"  📦 Best-Selling Product  : {top_prod}")
print(f"  📣 Best Marketing Channel: {best_channel}")
print(f"  💥 Highest Damage Category: {worst_cat_dmg}")
print(f"  🚴 On-Time Delivery Rate : {(1 - orders['is_late'].mean())*100:.1f}%")
print(f"  ⭐ Avg Customer Rating   : {feedback['rating'].mean():.2f} / 5.0")
print(f"  😊 Positive Sentiment    : {(feedback['sentiment']=='Positive').mean()*100:.1f}%")
print('=' * 60)
print('  ✅ Phase 2 Complete! Ready for Phase 3 — SQL Analysis')
print('=' * 60)
print(f'\n  📁 Charts saved to: outputs/charts/ (12 charts total)')